### SQL Homework
Use this notebook to answer the questions.
It can be in the same project as the previous homework.  
When you are ready, **upload** to your github repo, and send me the link (just zip a txt file with the repo's address, and upload it as the homework).  
If your repo is private, invite me: balazs.balogh@cubixedu.com.

#### Import the SparkSession, create it then load the taxi data (yellow_tripdata_2024-08.parquet)

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
import pyspark.sql.types as st

spark = (
    SparkSession
    .builder
    .appName("sql_homework")
    .master("local[*]")
    .getOrCreate()
)


taxi_df = (

    spark
    .read
    .format("parquet")
    .load("data/yellow_tripdata_2024-08.parquet")
)


#### 1. What is the total fare amount for all trips?  
Please round the answer to two decimal places.

In [3]:
# PySpark version
(
    taxi_df
    .agg(sf.format_number(sf.sum("fare_amount"), 2).alias("total_fare_amount"))
    .show(truncate=True)
    )



# Register the DataFrame as a temporary SQL view
taxi_df.createOrReplaceTempView("taxi")

# Run the SQL query
spark.sql("""
    SELECT FORMAT_NUMBER(SUM(fare_amount), 2) AS total_fare_amount
    FROM taxi
""").show(truncate=True)

+-----------------+
|total_fare_amount|
+-----------------+
|    58,750,998.06|
+-----------------+

+-----------------+
|total_fare_amount|
+-----------------+
|    58,750,998.06|
+-----------------+



#### 2. Show the maximum fare amount, minimum fare amount, and average fare amount for each payment type. Order by payment type.
Round where you need to two decimal places.

In [4]:
#Pyspark version
(
    taxi_df
    .groupBy("payment_type")
    .agg(
        sf.format_number(sf.max("fare_amount"), 2).alias("max_fare_amount")
        ,sf.format_number(sf.min("fare_amount"), 2).alias("min_fare_amount")
        ,sf.format_number(sf.avg("fare_amount"), 2).alias("avg_fare_amount")
        )
    .show(truncate=True)
    )

#SQL version
taxi_df.createOrReplaceTempView("taxi")

spark.sql("""
    SELECT payment_type,
           FORMAT_NUMBER(max(fare_amount),2) as max_fare_amount,
           FORMAT_NUMBER(min(fare_amount),2) as min_fare_amount,
           FORMAT_NUMBER(avg(fare_amount),2) as avg_fare_amount
    FROM taxi
    GROUP BY payment_type
    """).show(truncate=True)

+------------+---------------+---------------+---------------+
|payment_type|max_fare_amount|min_fare_amount|avg_fare_amount|
+------------+---------------+---------------+---------------+
|           1|         650.00|        -108.70|          20.60|
|           3|         999.00|        -999.00|           6.26|
|           2|       1,386.20|      -1,174.10|          19.09|
|           4|         900.00|        -900.00|           1.37|
|           0|         394.76|         -72.20|          19.62|
+------------+---------------+---------------+---------------+

+------------+---------------+---------------+---------------+
|payment_type|max_fare_amount|min_fare_amount|avg_fare_amount|
+------------+---------------+---------------+---------------+
|           1|         650.00|        -108.70|          20.60|
|           3|         999.00|        -999.00|           6.26|
|           2|       1,386.20|      -1,174.10|          19.09|
|           4|         900.00|        -900.00|        

#### 3. For trips with a fare amount greater than 20, what is the total tip amount for each day (based on the tpep_pickup_datetime)?
Round the tip to two decimal places, and order the results from highest total tip amount.  
Hint: Check DATE() function, to convert tpep_pickup_datetime to date, to get only the YYYY-MM-DD.

In [5]:
#Pyspark version
(
    taxi_df
    .filter(sf.col("fare_amount") > 20)
    .withColumn("pickup_date", sf.to_date(sf.col("tpep_pickup_datetime")))
    .groupBy("pickup_date")
    .agg(sf.format_number(sf.sum("tip_amount"), 2).alias("total_tip_amount"))
    .orderBy(sf.sum("tip_amount").desc())
    .show(truncate=True)
)




#SQL version
spark.sql("""
    SELECT 
        DATE(tpep_pickup_datetime) AS pickup_date,
        FORMAT_NUMBER(SUM(tip_amount), 2) AS total_tip_amount
    FROM 
        taxi
    WHERE 
        fare_amount > 20
    GROUP BY 
        DATE(tpep_pickup_datetime)
    ORDER BY 
        SUM(tip_amount) DESC
""").show(truncate=True)

+-----------+----------------+
|pickup_date|total_tip_amount|
+-----------+----------------+
| 2024-08-07|      200,170.81|
| 2024-08-01|      197,694.16|
| 2024-08-29|      192,837.64|
| 2024-08-08|      192,807.33|
| 2024-08-19|      184,872.55|
| 2024-08-22|      182,233.24|
| 2024-08-30|      180,368.43|
| 2024-08-28|      180,220.74|
| 2024-08-05|      176,271.55|
| 2024-08-15|      176,223.00|
| 2024-08-06|      175,451.67|
| 2024-08-11|      174,769.89|
| 2024-08-20|      173,310.48|
| 2024-08-25|      171,774.94|
| 2024-08-04|      169,589.92|
| 2024-08-23|      168,495.10|
| 2024-08-02|      167,908.50|
| 2024-08-14|      166,266.40|
| 2024-08-21|      165,647.98|
| 2024-08-27|      164,666.80|
+-----------+----------------+
only showing top 20 rows

+-----------+----------------+
|pickup_date|total_tip_amount|
+-----------+----------------+
| 2024-08-07|      200,170.81|
| 2024-08-01|      197,694.16|
| 2024-08-29|      192,837.64|
| 2024-08-08|      192,807.33|
| 2024-08-19|

#### 4. For each trip, show the fare amount along with a column that indicates if the trip was "expensive" (greater than 30) or "cheap" (less than or equal to 30).
Hint: Use CASE WHEN for deciding on expensive, or cheap.

In [6]:
#Pyspark version
(
    taxi_df
    .select(
        "fare_amount",
        sf.when(sf.col("fare_amount") > 30, "expensive")
          .otherwise("cheap")
          .alias("trip_category")
    )
    .show()
)

#SQL version
spark.sql("""
    SELECT 
        fare_amount,
        CASE
            WHEN fare_amount > 30 THEN 'expensive'
            ELSE 'cheap'
        END AS trip_category
    FROM 
        taxi
""").show()

+-----------+-------------+
|fare_amount|trip_category|
+-----------+-------------+
|       28.9|        cheap|
|       40.8|    expensive|
|       52.0|    expensive|
|       17.0|        cheap|
|        5.1|        cheap|
|        5.8|        cheap|
|       16.3|        cheap|
|       17.7|        cheap|
|       27.5|        cheap|
|       11.4|        cheap|
|        9.3|        cheap|
|        5.1|        cheap|
|       16.3|        cheap|
|       45.0|    expensive|
|        8.6|        cheap|
|        7.2|        cheap|
|       13.5|        cheap|
|       14.9|        cheap|
|       10.7|        cheap|
|        8.6|        cheap|
+-----------+-------------+
only showing top 20 rows

+-----------+-------------+
|fare_amount|trip_category|
+-----------+-------------+
|       28.9|        cheap|
|       40.8|    expensive|
|       52.0|    expensive|
|       17.0|        cheap|
|        5.1|        cheap|
|        5.8|        cheap|
|       16.3|        cheap|
|       17.7|        c

#### 5. Find the first trip (based on tpep_pickup_datetime) for each VendorID and display the fare amount.
Hint: You can use CTE with ROW_NUMBER().

In [7]:
from pyspark.sql.window import Window

#Pyspark version
window_spec = Window.partitionBy("VendorID").orderBy("tpep_pickup_datetime")


(
    taxi_df
    .withColumn("row_num", sf.row_number().over(window_spec))
    .filter(sf.col("row_num") == 1)
    .select("VendorID", "tpep_pickup_datetime", "fare_amount")
    .orderBy("VendorID")
    .show()
)



#SQL version
spark.sql("""
    WITH RankedTrips AS (
        SELECT 
            VendorID,
            tpep_pickup_datetime,
            fare_amount,
            ROW_NUMBER() OVER (PARTITION BY VendorID ORDER BY tpep_pickup_datetime) AS row_num
        FROM 
            taxi
    )
    SELECT 
        VendorID,
        tpep_pickup_datetime,
        fare_amount
    FROM 
        RankedTrips
    WHERE 
        row_num = 1
    ORDER BY
        VendorID
""").show()

+--------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|fare_amount|
+--------+--------------------+-----------+
|       1| 2024-08-01 00:00:04|       70.0|
|       2| 2009-01-01 00:02:52|       10.7|
+--------+--------------------+-----------+

+--------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|fare_amount|
+--------+--------------------+-----------+
|       1| 2024-08-01 00:00:04|       70.0|
|       2| 2009-01-01 00:02:52|       10.7|
+--------+--------------------+-----------+



#### 7. Calculate the average trip distance for each VendorID, and assign a label of 'Above Average' or 'Below Average' for each trip based on the distance relative to the VendorID’s average trip distance.
Hint: CTE joined back to the main DataFrame.

In [39]:
taxi_df.show(3)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-08-01 00:21:00|  2024-08-01 00:36:13|              1|          7.4|         1|                 N|         138|          80|           1|       28.9| 7.75|    0.5|      7.6

In [44]:
# Check basic statistics of the trip_distance column
taxi_df.select("trip_distance").describe().show()

# Check for number of zero values
taxi_df.filter(sf.col("trip_distance") == 0).count()

+-------+------------------+
|summary|     trip_distance|
+-------+------------------+
|  count|           2979183|
|   mean|  4.94486408522048|
| stddev|378.52743052165886|
|    min|               0.0|
|    max|         327025.19|
+-------+------------------+



57413

In [38]:
spark.sql("""
    WITH VendorAverages AS (
        SELECT 
            VendorID,
            AVG(trip_distance) AS avg_distance
        FROM 
            taxi
        GROUP BY 
            VendorID
    )
    
    SELECT 
        t.VendorID,
        t.trip_distance,
        v.avg_distance,
        CASE
            WHEN t.trip_distance > v.avg_distance THEN 'Above Average'
            ELSE 'Below Average'
        END AS distance_category
    FROM 
        taxi t
    JOIN 
        VendorAverages v ON t.VendorID = v.VendorID
    ORDER BY
        t.VendorID, t.trip_distance
""").show()

+--------+-------------+-----------------+-----------------+
|VendorID|trip_distance|     avg_distance|distance_category|
+--------+-------------+-----------------+-----------------+
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|4.034096101093374|    Below Average|
|       1|          0.0|

In [40]:
spark.sql("""

        SELECT 
            VendorID,
            AVG(trip_distance) AS avg_distance
        FROM 
            taxi
        GROUP BY 
            VendorID
    """).show(5)

+--------+-----------------+
|VendorID|     avg_distance|
+--------+-----------------+
|       1|4.034096101093374|
|       2|5.214354531064327|
+--------+-----------------+



In [45]:
spark.sql("""
    WITH VendorAverages AS (
        SELECT 
            VendorID,
            AVG(trip_distance) AS avg_distance
        FROM 
            taxi
        GROUP BY 
            VendorID
    )
    
    SELECT 
        t.VendorID,
        t.trip_distance,
        v.avg_distance,
        CASE
            WHEN t.trip_distance > v.avg_distance THEN 'Above Average'
            WHEN t.trip_distance = v.avg_distance THEN 'Equal to Average'
            ELSE 'Below Average'
        END AS distance_category
    FROM 
        taxi t
    JOIN 
        VendorAverages v ON t.VendorID = v.VendorID
    WHERE t.trip_distance > 0  -- Filter out zero-distance trips
    ORDER BY
        t.VendorID, t.trip_distance
    LIMIT 20  -- Get a sample with some variation
""").show()

+--------+-------------+-----------------+-----------------+
|VendorID|trip_distance|     avg_distance|distance_category|
+--------+-------------+-----------------+-----------------+
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|4.034096101093374|    Below Average|
|       1|          0.1|